In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. CARREGAMENTO E PREPARAÇÃO DOS DADOS
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

precos = pd.read_csv(os.path.join(pasta_dados, "precos_sanitizada_v3.csv"))
cdi = pd.read_excel(os.path.join(pasta_dados, "CDI_2010_2026.xlsx"))
ibov = pd.read_excel(os.path.join(pasta_dados, "IBOV_2010_2026.xlsx"))

precos['Data'] = pd.to_datetime(precos['Data'])
cdi['Data'] = pd.to_datetime(cdi['Data'])
ibov['Data'] = pd.to_datetime(ibov['Data'])

# Cálculo dos retornos simples diários
retornos_ativos = precos.set_index('Data').pct_change().dropna(how='all')
retornos_ibov = ibov.set_index('Data').pct_change().dropna()
retornos_ibov.columns = ['IBOV']

# Alinhamento temporal
df_merged = retornos_ativos.merge(retornos_ibov, left_index=True, right_index=True)
df_merged = df_merged.merge(cdi.set_index('Data'), left_index=True, right_index=True)

# ==========================================
# 2. CÁLCULO DO DELTA (AVERSÃO AO RISCO DO MERCADO)
# ==========================================
# Prêmio de risco diário do Ibovespa em relação ao CDI
excesso_ibov = df_merged['IBOV'] - df_merged['CDI']

# Delta = Média do excesso de retorno / Variância do mercado
# Usamos os dados diários e multiplicamos por 252 para anualizar o prêmio e a variância
retorno_anual_excesso = excesso_ibov.mean() * 252
variancia_anual_ibov = excesso_ibov.var() * 252

delta = retorno_anual_excesso / variancia_anual_ibov
print(f"Coeficiente Delta de Aversão ao Risco do Mercado: {round(delta, 4)}")

# ==========================================
# 3. MATRIZ DE COVARIÂNCIA ANUALIZADA (Σ)
# ==========================================
# Calculamos apenas para as ações da base
colunas_acoes = [col for col in retornos_ativos.columns]
Sigma = df_merged[colunas_acoes].cov() * 252

# ==========================================
# 4. PESOS DE MERCADO (w_mkt)
# ==========================================
# No seu TCC real, você importará o valor de mercado (Market Cap) via yfinance.
# Para este exemplo estrutural, vamos simular pesos iguais que somam 100% (1.0).
# O vetor de pesos precisa ter o mesmo tamanho (shape) que o número de ações.
num_ativos = len(colunas_acoes)
pesos_mkt = np.repeat(1.0 / num_ativos, num_ativos) 

# ==========================================
# 5. CÁLCULO DOS RETORNOS IMPLÍCITOS (Π)
# ==========================================
# Multiplicação matricial: Π = delta * Sigma * w_mkt
pi_anualizado = delta * np.dot(Sigma, pesos_mkt)

# Criar um DataFrame final para o seu relatório
df_equilibrio = pd.DataFrame({
    'Ativo': colunas_acoes,
    'Peso Mkt Estimado': pesos_mkt,
    'Retorno Implicito Equilíbrio (Π)': [f"{round(r * 100, 2)}%" for r in pi_anualizado]
})

print("\n--- Vetor de Equilíbrio Neutro (Prior de Black-Litterman) ---")
print(df_equilibrio)

Coeficiente Delta de Aversão ao Risco do Mercado: -0.0884

--- Vetor de Equilíbrio Neutro (Prior de Black-Litterman) ---


NameError: name 'df_conducao' is not defined